# FinFlow - Account Takeover Investigation

Risk Data Scientist take-home. The scenario: a customer says their FinFlow
account was taken over and they're seeing vendor payments they never made.

This notebook works through it in order:

| Section | Question it answers |
|---|---|
| 0 | Setup and load |
| 1 | Can I trust this data? |
| 2 | What do the three label fields actually mean? |
| 3 | What does normal activity look like? |
| 4 | Who got hit? |
| 5 | When did it happen? |
| 6 | What does the fraud look like? |
| 7 | How did they get in? |
| 8 | What can a model actually use? |

Run top to bottom on a clean kernel.

## 0. Setup

Nothing clever here. Fix the seed so re-running gives the same numbers as the
slides, use relative paths so this runs on someone else's machine, and make a
folder for figures.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

SEED = 42
np.random.seed(SEED)

DATA = Path("finflow_case_study_data")
FIGDIR = Path("outputs/figures")
FIGDIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110


def savefig(name):
    """Save the current figure to outputs/figures so slides can use it."""
    plt.tight_layout()
    plt.savefig(FIGDIR / f"{name}.png", dpi=150, bbox_inches="tight")


print("pandas", pd.__version__, "| numpy", np.__version__)

pandas 1.3.4 | numpy 1.24.3


### Load

Two CSVs. Parse the timestamp columns at read time so I never have to remember
to convert them later - a string that looks like a date but sorts like text is
a classic way to get a wrong answer.

In [3]:
txn = pd.read_csv(
    DATA / "transactions.csv",
    parse_dates=["txn_ts", "vendor_add_date"],
)
log = pd.read_csv(
    DATA / "logins.csv",
    parse_dates=["login_ts"],
)

print(f"transactions: {txn.shape[0]:,} rows x {txn.shape[1]} cols")
print(f"logins:       {log.shape[0]:,} rows x {log.shape[1]} cols")
print()
print("transactions cover", txn.txn_ts.min().date(), "to", txn.txn_ts.max().date())
print("logins cover      ", log.login_ts.min().date(), "to", log.login_ts.max().date())
txn.head(3)

transactions: 184,432 rows x 16 cols
logins:       72,621 rows x 12 cols

transactions cover 2026-04-01 to 2026-06-30
logins cover       2026-04-01 to 2026-06-29


,txn_id,org_id,user_id,txn_ts,amount,currency,transfer_type,vendor_id,vendor_add_date,vendor_bank_country,is_new_bank_account,approval_bypassed,session_id,dispute_flag,ops_review_result,txn_fraud_label
0,t_00140996,org_0062,u_0062_02,2026-04-01 01:31:27,433.07,USD,virtual card,v_0062_001,2024-01-04 00:00:00,US,False,False,s_00066979,False,NaN,False
1,t_00164638,org_0005,u_0005_04,2026-04-01 01:32:22,14364.96,USD,ACH,v_0005_015,2026-04-02 20:04:30,US,False,False,s_00063504,False,NaN,False
2,t_00060959,org_0005,u_0005_04,2026-04-01 01:34:22,2064.12,USD,wire,v_0005_004,2026-04-04 20:07:35,US,True,False,s_00063504,False,NaN,False


In [4]:
log.head(3)

,session_id,user_id,org_id,login_ts,device_id,ip_country,ip_region,is_vpn_or_proxy,is_new_device,mfa_triggered,mfa_result,login_success
0,s_00068712,u_0052_02,org_0052,2026-04-01 01:16:05,d_a85519273d4d,US,WA,False,False,False,n.a.,True
1,s_00066979,u_0062_02,org_0062,2026-04-01 01:16:27,d_a32804edac33,US,WA,False,False,False,n.a.,True
2,s_00019444,u_0021_02,org_0021,2026-04-01 01:26:39,d_b4a630595796,US,VA,False,False,False,n.a.,True


## 1. Can I trust this data?

Before any analysis, three questions:

1. **Is anything missing?** Nulls in the wrong place change what a number means.
2. **Is anything duplicated?** A duplicated transaction would inflate every count.
3. **Does the join actually work?** The brief says the files join on `session_id`.
   If that join silently drops rows, every number after it is wrong and I'd
   never notice.

This is the boring section that stops me from presenting a wrong headline.

In [5]:
print("=== column types ===")
print(txn.dtypes.to_string())
print()
print(log.dtypes.to_string())

=== column types ===
txn_id                         object
org_id                         object
user_id                        object
txn_ts                 datetime64[ns]
amount                        float64
currency                       object
transfer_type                  object
vendor_id                      object
vendor_add_date        datetime64[ns]
vendor_bank_country            object
is_new_bank_account              bool
approval_bypassed                bool
session_id                     object
dispute_flag                     bool
ops_review_result              object
txn_fraud_label                  bool

session_id                 object
user_id                    object
org_id                     object
login_ts           datetime64[ns]
device_id                  object
ip_country                 object
ip_region                  object
is_vpn_or_proxy              bool
is_new_device                bool
mfa_triggered                bool
mfa_result                 obj

In [6]:
print("=== missing values (only columns that have any) ===")
for name, d in [("transactions", txn), ("logins", log)]:
    n = d.isna().sum()
    n = n[n > 0]
    if len(n) == 0:
        print(f"{name}: no missing values")
    else:
        print(f"{name}:")
        for col, cnt in n.items():
            print(f"   {col}: {cnt:,} missing ({cnt / len(d):.1%})")

=== missing values (only columns that have any) ===
transactions:
   ops_review_result: 182,269 missing (98.8%)
logins: no missing values


`ops_review_result` is the only column with missing values, and it's missing
for **98.8%** of transactions.

That's not a data quality problem - it's the point. FinFlow's ops team only
reviews a small slice of transactions. A null there means *"nobody looked at
this"*, not *"this was fine"*. I come back to it in Section 2, because it's the
single easiest way to get this case study wrong.

In [7]:
print("=== duplicates ===")
print("duplicate txn_id:              ", txn.txn_id.duplicated().sum())
print("duplicate session_id in logins:", log.session_id.duplicated().sum())
print("fully duplicated txn rows:     ", txn.duplicated().sum())
print("fully duplicated login rows:   ", log.duplicated().sum())

=== duplicates ===
duplicate txn_id:               0
duplicate session_id in logins: 0
fully duplicated txn rows:      0
fully duplicated login rows:    0


### The join

A transaction happens inside a login session. So I attach each transaction's
session to it with a **left** join from transactions - left, because I want to
keep every transaction even if its session were missing.

Then I check it instead of trusting it:

In [8]:
df = txn.merge(log, on="session_id", how="left", suffixes=("", "_login"))

# 1. the join must not add or drop rows
assert len(df) == len(txn), f"row count changed: {len(txn):,} -> {len(df):,}"

# 2. every transaction must have found a session
orphans = df.login_ts.isna().sum()
assert orphans == 0, f"{orphans} transactions have no matching login session"

# 3. the two files must agree on who did what
assert (df.user_id == df.user_id_login).all(), "user_id disagrees across the join"
assert (df.org_id == df.org_id_login).all(), "org_id disagrees across the join"

df = df.drop(columns=["user_id_login", "org_id_login"])

sessions_no_txn = (~log.session_id.isin(txn.session_id)).sum()
print(f"joined cleanly: {len(df):,} rows, {df.shape[1]} columns")
print(f"login sessions that produced no transaction: {sessions_no_txn:,}")

joined cleanly: 184,432 rows, 25 columns
login sessions that produced no transaction: 1,944


Clean. Every transaction maps to a real session, and the two files agree on
the user and the org, so I dropped the duplicated columns.

**1,944 sessions produced no transaction.** That's normal - people log in just
to look at things. But it also means the login file contains activity the
transaction file can't see, which matters in Section 7 when I go looking for
how the attacker got in.

I used `assert` instead of printing a warning on purpose. If this notebook gets
re-run on different data and the join breaks, it should stop, not quietly
produce a wrong slide.

### One thing that is genuinely weird

`vendor_add_date` is supposed to be when the vendor was added to the org's
list. So it should always land *before* any payment to that vendor.

In [9]:
backwards = df.vendor_add_date > df.txn_ts
print(f"payments dated BEFORE their vendor was added: {backwards.sum():,} "
      f"({backwards.mean():.1%} of all transactions)")
print(f"   fraud rate in those rows:  {df.loc[backwards, 'txn_fraud_label'].mean():.4%}")
print(f"   fraud rate everywhere else: {df.loc[~backwards, 'txn_fraud_label'].mean():.4%}")

vendor_age_days = (df.txn_ts - df.vendor_add_date).dt.total_seconds() / 86400
print("\nvendor age at payment time, in days:")
print(vendor_age_days.describe().round(1).to_string())

payments dated BEFORE their vendor was added: 38,625 (20.9% of all transactions)
   fraud rate in those rows:  0.0078%
   fraud rate everywhere else: 0.4492%

vendor age at payment time, in days:
count    184432.0
mean        274.9
std         313.3
min         -89.7
25%           7.5
50%         161.2
75%         514.8
max         986.9


**21% of transactions were paid to a vendor that, according to this field,
didn't exist yet.**

My read: `vendor_add_date` records the *most recent change* to the vendor
record - the brief says bank details can be changed later - not the original
add date. So for a fifth of the data it's a "last modified" timestamp wearing
an "added on" name.

Why it matters: the obvious feature here is *"how new is this vendor?"*, and
that feature comes out **negative** for 21% of rows. Fed in raw, the model
would be learning from a field that means two different things depending on
the row.

**Assumption I'm making:** I don't trust `vendor_age_days` on its own. I lean
on `is_new_bank_account` instead, which the brief defines cleanly (bank details
added or changed in the last 7 days), and I keep the negative-age rows as a
flag rather than dropping them.

Worth noting the fraud rate in those rows is basically zero, so this quirk is
not where the fraud is hiding. It's a data-definition problem, not a lead.

**Section 1 takeaway:** the data joins cleanly and has no duplicates. Two
things to carry forward - `ops_review_result` is missing 98.8% of the time by
design, and `vendor_add_date` doesn't mean what its name says.

In [16]:
log[(log["device_id"] == "d_5019b1fc56a9") & (log["login_ts"] <= "2026-04-24 20:20:00")]

,session_id,user_id,org_id,login_ts,device_id,ip_country,ip_region,is_vpn_or_proxy,is_new_device,mfa_triggered,mfa_result,login_success
17945,s_00043805,u_0001_01,org_0001,2026-04-22 20:20:00,d_5019b1fc56a9,SG,Central,False,True,True,failed,False
17948,s_00060060,u_0001_01,org_0001,2026-04-22 20:21:00,d_5019b1fc56a9,PH,Metro Manila,False,False,True,failed,False
17955,s_00070713,u_0001_01,org_0001,2026-04-22 20:23:00,d_5019b1fc56a9,PH,Metro Manila,False,False,True,failed,False
17957,s_00028683,u_0001_01,org_0001,2026-04-22 20:24:00,d_5019b1fc56a9,SG,Central,False,False,True,failed,False
17963,s_00047620,u_0001_01,org_0001,2026-04-22 20:28:00,d_5019b1fc56a9,SG,Central,False,False,True,failed,False
...,...,...,...,...,...,...,...,...,...,...,...,...
19452,s_00044442,u_0001_01,org_0001,2026-04-24 07:47:28,d_5019b1fc56a9,SG,Central,False,False,False,n.a.,True
19455,s_00035569,u_0001_01,org_0001,2026-04-24 07:58:47,d_5019b1fc56a9,US,PA,True,False,True,passed,True
19456,s_00068808,u_0101_01,org_0101,2026-04-24 08:00:07,d_5019b1fc56a9,US,CO,True,False,False,n.a.,True
19461,s_00013791,u_0001_01,org_0001,2026-04-24 08:15:35,d_5019b1fc56a9,SG,Central,False,False,True,passed,True


In [22]:
fraud_sessions = set(txn.loc[txn.txn_fraud_label, "session_id"])
log["attack"] = log.session_id.isin(fraud_sessions)

# The devices those sessions ran on -- found, not assumed.
ATTACKER_DEVICES = sorted(log.loc[log.attack, "device_id"].unique())
log["att_dev"] = log.device_id.isin(ATTACKER_DEVICES)

atk   = log[log.attack]                   # 122 sessions that got in and moved money
recon = log[log.att_dev & ~log.attack]    #  46 sessions that were blocked
norm  = log[~log.att_dev]       

In [23]:
recon_test = log[log.device_id.isin(["d_5019b1fc56a9", "d_93bba37589da"])
            & ~log.login_success].sort_values("login_ts")
recon_test.head()

,session_id,user_id,org_id,login_ts,device_id,ip_country,ip_region,is_vpn_or_proxy,is_new_device,mfa_triggered,mfa_result,login_success,attack,att_dev
17945,s_00043805,u_0001_01,org_0001,2026-04-22 20:20:00,d_5019b1fc56a9,SG,Central,False,True,True,failed,False,False,True
17948,s_00060060,u_0001_01,org_0001,2026-04-22 20:21:00,d_5019b1fc56a9,PH,Metro Manila,False,False,True,failed,False,False,True
17955,s_00070713,u_0001_01,org_0001,2026-04-22 20:23:00,d_5019b1fc56a9,PH,Metro Manila,False,False,True,failed,False,False,True
17957,s_00028683,u_0001_01,org_0001,2026-04-22 20:24:00,d_5019b1fc56a9,SG,Central,False,False,True,failed,False,False,True
17963,s_00047620,u_0001_01,org_0001,2026-04-22 20:28:00,d_5019b1fc56a9,SG,Central,False,False,True,failed,False,False,True


In [24]:
log = pd.read_csv('finflow_case_study_data/logins.csv', parse_dates=['login_ts'])
DEV = ['d_5019b1fc56a9', 'd_93bba37589da']
log['bad'] = log.device_id.isin(DEV)
log

,session_id,user_id,org_id,login_ts,device_id,ip_country,ip_region,is_vpn_or_proxy,is_new_device,mfa_triggered,mfa_result,login_success,bad
0,s_00068712,u_0052_02,org_0052,2026-04-01 01:16:05,d_a85519273d4d,US,WA,False,False,False,n.a.,True,False
1,s_00066979,u_0062_02,org_0062,2026-04-01 01:16:27,d_a32804edac33,US,WA,False,False,False,n.a.,True,False
2,s_00019444,u_0021_02,org_0021,2026-04-01 01:26:39,d_b4a630595796,US,VA,False,False,False,n.a.,True,False
3,s_00063504,u_0005_04,org_0005,2026-04-01 01:30:22,d_44631cb993f2,US,NC,False,False,False,n.a.,True,False
4,s_00015370,u_0062_02,org_0062,2026-04-01 01:34:33,d_a32804edac33,US,WA,False,False,False,n.a.,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
72616,s_00036801,u_0069_03,org_0069,2026-06-29 22:53:04,d_7c3ecfc84635,US,CO,False,False,False,n.a.,True,False
72617,s_00034527,u_0008_02,org_0008,2026-06-29 22:53:05,d_dafffcd2cfc0,US,MA,False,False,False,n.a.,True,False
72618,s_00025807,u_0062_07,org_0062,2026-06-29 22:54:59,d_f3b364e063ee,US,WA,False,False,False,n.a.,True,False
72619,s_00052195,u_0056_06,org_0056,2026-06-29 22:57:15,d_e0731f7026b3,US,MN,False,False,False,n.a.,True,False


In [25]:
import pandas as pd

txn = pd.read_csv("finflow_case_study_data/transactions.csv", parse_dates=["txn_ts"])
log = pd.read_csv("finflow_case_study_data/logins.csv",       parse_dates=["login_ts"])

# A session is an "attack" session if it moved money that was later
# adjudicated fraud. This is the ONLY place the label is used.
fraud_sessions = set(txn.loc[txn.txn_fraud_label, "session_id"])
log["attack"] = log.session_id.isin(fraud_sessions)

# The devices those sessions ran on -- found, not assumed.
ATTACKER_DEVICES = sorted(log.loc[log.attack, "device_id"].unique())
log["att_dev"] = log.device_id.isin(ATTACKER_DEVICES)

atk   = log[log.attack]                   # 122 sessions that got in and moved money
recon = log[log.att_dev & ~log.attack]    #  46 sessions that were blocked
norm  = log[~log.att_dev]                 # 72,453 sessions on every other device

ATTACKER_DEVICES
# ['d_5019b1fc56a9', 'd_93bba37589da']      <- 2 of 728 devices

log.loc[log.att_dev, "user_id"].nunique()   # 5   (all seat _01)
log.loc[log.att_dev, "org_id"].nunique()    # 5   (of 110)
log.attack.sum()                            # 122 got in
(log.att_dev & ~log.attack).sum()           # 46  blocked

46